# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/i-am-gk/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule

Prioritize pages that are both **stale** and have meaningful organic visibility. A page receives a higher score when it has not been updated for a long time and has enough impressions to represent a potentially actionable refresh opportunity.

### Reason codes

- `stale_visible` — page is stale and has meaningful impressions.
- `stale_low_visibility` — page is stale but has low impressions.
- `fresh_visible` — page has meaningful impressions but is not stale.
- `other` — page does not meet the main conditions.

### Action labels

- `refresh_review` — review the page for a possible content refresh.
- `monitor` — keep under observation rather than prioritizing immediately.

The score is a transparent rule-based baseline, not a trained model. It is intended for directional decision-support only.

In [18]:
import pandas as pd
from pathlib import Path

# Load the starter dataset
csv_path = Path("/content/content_refresh_anonymized.csv")

if not csv_path.exists():
    matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))
    if matches:
        csv_path = matches[0]
    else:
        raise FileNotFoundError(
            "Upload content_refresh_anonymized.csv to the Colab Files panel first."
        )

df = pd.read_csv(csv_path)

# Basic checks
required_cols = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Transparent rule thresholds
STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 500

# Signal checks
df["is_stale"] = df["days_since_last_update"] >= STALE_DAYS
df["is_visible"] = df["impressions_90d"] >= VISIBLE_IMPRESSIONS

print("Rows:", len(df))
print("Stale pages:", int(df["is_stale"].sum()))
print("Visible pages:", int(df["is_visible"].sum()))
print("Stale + visible pages:", int((df["is_stale"] & df["is_visible"]).sum()))

print("\nThresholds:")
print("Stale:", f">= {STALE_DAYS} days")
print("Visible:", f">= {VISIBLE_IMPRESSIONS} impressions over 90 days")

Rows: 30000
Stale pages: 174
Visible pages: 16726
Stale + visible pages: 17

Thresholds:
Stale: >= 180 days
Visible: >= 500 impressions over 90 days


### Ranked baseline queue

The baseline gives the highest priority to pages that are both stale and visible.

Score:
- 2 points = stale + visible
- 1 point = stale only
- 0 points = all other pages

The score is intentionally simple and transparent. It does not use `trend_direction` or `trend_pct`, because those fields are label-derived and would leak the outcome into the baseline.

Action:
- `refresh_review` for stale + visible pages
- `monitor` for all other pages

Reason codes describe why a page received its score.

In [19]:
# ML-07 Section 2 — Build ranked queue

# Score the two pre-defined signals.
df["baseline_score"] = (
    df["is_stale"].astype(int) +
    df["is_visible"].astype(int)
)

# Assign reason codes.
df["reason_code"] = "other"

df.loc[
    df["is_stale"] & df["is_visible"],
    "reason_code"
] = "stale_visible"

df.loc[
    df["is_stale"] & ~df["is_visible"],
    "reason_code"
] = "stale_low_visibility"

df.loc[
    ~df["is_stale"] & df["is_visible"],
    "reason_code"
] = "fresh_visible"

# Assign actions.
df["action"] = "monitor"

df.loc[
    df["reason_code"] == "stale_visible",
    "action"
] = "refresh_review"

# Rank by score first, then by impressions as a transparent tie-breaker.
ranked = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

ranked["rank"] = range(1, len(ranked) + 1)

# Keep useful fields in the output.
output_cols = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
]

queue = ranked[output_cols].copy()

# Write the required CSV.
output_path = Path("/content/baseline_action_score.csv")
queue.to_csv(output_path, index=False)

print("Ranked rows:", len(queue))
print("Output file:", output_path)
print("\nScore distribution:")
print(queue["baseline_score"].value_counts().sort_index(ascending=False))

print("\nReason-code distribution:")
print(queue["reason_code"].value_counts())

print("\nTop 20:")
display(queue.head(20)) # Save the required assignment output path inside the repository structure.
repo_output_dir = Path("/content/work/outputs")
repo_output_dir.mkdir(parents=True, exist_ok=True)

repo_output_path = repo_output_dir / "baseline_action_score.csv"
queue.to_csv(repo_output_path, index=False)

print("Assignment output:", repo_output_path)

Ranked rows: 30000
Output file: /content/baseline_action_score.csv

Score distribution:
baseline_score
2       17
1    16866
0    13117
Name: count, dtype: int64

Reason-code distribution:
reason_code
fresh_visible           16709
other                   13117
stale_low_visibility      157
stale_visible              17
Name: count, dtype: int64

Top 20:


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,trend_direction
16751,1,content_cf56e2e2e282,client_7f2253d7e2,2,stale_visible,refresh_review,194,61678,down
16514,2,content_7368877ea310,client_7f2253d7e2,2,stale_visible,refresh_review,194,59472,down
7021,3,content_1bfaa38ff26c,client_7f2253d7e2,2,stale_visible,refresh_review,194,25715,down
21268,4,content_0a91db491d14,client_7f2253d7e2,2,stale_visible,refresh_review,193,13299,down
11489,5,content_5feee3994adb,client_7f2253d7e2,2,stale_visible,refresh_review,194,7812,down
12045,6,content_c2d929d83eaa,client_7f2253d7e2,2,stale_visible,refresh_review,193,7558,down
698,7,content_b16bd7307b39,client_7f2253d7e2,2,stale_visible,refresh_review,194,4590,down
5327,8,content_fe16a55cd13d,client_7f2253d7e2,2,stale_visible,refresh_review,194,4556,down
26810,9,content_ecb6215e79fd,client_7f2253d7e2,2,stale_visible,refresh_review,194,4429,down
20837,10,content_928af3e22c80,client_7f2253d7e2,2,stale_visible,refresh_review,193,1697,down


Assignment output: /content/work/outputs/baseline_action_score.csv


### Signal checks and Top-20 review

Two simple signals are checked before relying on the baseline:

1. **Staleness:** whether a page has been without an update for at least 180 days. This is linked to the refresh/staleness decision.
2. **Organic visibility:** whether a page has at least 500 impressions over the trailing 90-day window. This is linked to prioritizing pages with meaningful search visibility.

The bucket tables below compare the observed share of `down` pages across signal buckets. The verdicts are descriptive:
- **CONFIRMED** = the observed pattern supports the signal.
- **OPPOSITE** = the observed pattern moves in the opposite direction.
- **MIXED** = the pattern is not consistent across buckets.
- **FALSE** = there is no useful directional separation.

The top-20 review is a manual-style review of the highest-ranked rows. Confidence notes describe how directly the row matches the rule, while the final note records what could make the recommendation wrong.

In [20]:
# ML-07 Section 3 — Signal checks + Top-20 review

review_df = df.copy()

# Create simple buckets for the two signals.
review_df["staleness_bucket"] = pd.cut(
    review_df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30d", "31-90d", "91-180d", "181d+"]
)

review_df["visibility_bucket"] = pd.cut(
    review_df["impressions_90d"],
    bins=[-1, 99, 499, 2999, float("inf")],
    labels=["0-99", "100-499", "500-2999", "3000+"]
)

# Define the observed outcome for evaluation only.
# This is NOT used by the baseline score.
review_df["is_down"] = review_df["trend_direction"].eq("down")

print("=== SIGNAL 1: STALENESS ===")
staleness_table = (
    review_df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        down_n=("is_down", "sum"),
        down_rate=("is_down", "mean")
    )
    .reset_index()
)

staleness_table["down_rate_pct"] = (
    staleness_table["down_rate"] * 100
).round(1)

display(
    staleness_table[
        ["staleness_bucket", "n", "down_n", "down_rate_pct"]
    ]
)

# Determine a descriptive verdict.
staleness_rates = staleness_table["down_rate"].dropna().tolist()

if len(staleness_rates) >= 2:
    if staleness_rates[-1] > staleness_rates[0]:
        staleness_verdict = "CONFIRMED"
    elif staleness_rates[-1] < staleness_rates[0]:
        staleness_verdict = "OPPOSITE"
    else:
        staleness_verdict = "MIXED"
else:
    staleness_verdict = "FALSE"

print("Staleness verdict:", staleness_verdict)


print("\n=== SIGNAL 2: ORGANIC VISIBILITY ===")
visibility_table = (
    review_df.groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        down_n=("is_down", "sum"),
        down_rate=("is_down", "mean")
    )
    .reset_index()
)

visibility_table["down_rate_pct"] = (
    visibility_table["down_rate"] * 100
).round(1)

display(
    visibility_table[
        ["visibility_bucket", "n", "down_n", "down_rate_pct"]
    ]
)

visibility_rates = visibility_table["down_rate"].dropna().tolist()

if len(visibility_rates) >= 2:
    if visibility_rates[-1] > visibility_rates[0]:
        visibility_verdict = "CONFIRMED"
    elif visibility_rates[-1] < visibility_rates[0]:
        visibility_verdict = "OPPOSITE"
    else:
        visibility_verdict = "MIXED"
else:
    visibility_verdict = "FALSE"

print("Visibility verdict:", visibility_verdict)


# ---------------------------------------------------------
# TOP-20 REVIEW
# ---------------------------------------------------------

top20 = ranked.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda r:
        "High: matches both baseline conditions."
        if r["reason_code"] == "stale_visible"
        else
        "Medium: high visibility but does not meet the staleness condition.",
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    lambda r:
        "The recorded update date may not reflect a recent refresh, or the page may not be actionable despite its visibility."
        if r["reason_code"] == "stale_visible"
        else
        "High impressions alone may not mean a refresh is needed; the page may already be performing acceptably or have another reason to remain unchanged.",
    axis=1
)

print("\n=== TOP-20 REVIEW ===")

review_cols = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "confidence_note",
    "what_would_make_it_wrong",
]

display(top20[review_cols])

=== SIGNAL 1: STALENESS ===


,staleness_bucket,n,down_n,down_rate_pct
0,0-30d,20480,10473,51.1
1,31-90d,175,103,58.9
2,91-180d,9171,5604,61.1
3,181d+,174,82,47.1


Staleness verdict: OPPOSITE

=== SIGNAL 2: ORGANIC VISIBILITY ===


,visibility_bucket,n,down_n,down_rate_pct
0,0-99,7994,3110,38.9
1,100-499,5280,3191,60.4
2,500-2999,8443,5240,62.1
3,3000+,8283,4721,57.0


Visibility verdict: CONFIRMED

=== TOP-20 REVIEW ===


,rank,content_id,action,reason_code,days_since_last_update,impressions_90d,trend_direction,confidence_note,what_would_make_it_wrong
16751,1,content_cf56e2e2e282,refresh_review,stale_visible,194,61678,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
16514,2,content_7368877ea310,refresh_review,stale_visible,194,59472,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
7021,3,content_1bfaa38ff26c,refresh_review,stale_visible,194,25715,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
21268,4,content_0a91db491d14,refresh_review,stale_visible,193,13299,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
11489,5,content_5feee3994adb,refresh_review,stale_visible,194,7812,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
12045,6,content_c2d929d83eaa,refresh_review,stale_visible,193,7558,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
698,7,content_b16bd7307b39,refresh_review,stale_visible,194,4590,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
5327,8,content_fe16a55cd13d,refresh_review,stale_visible,194,4556,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
26810,9,content_ecb6215e79fd,refresh_review,stale_visible,194,4429,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...
20837,10,content_928af3e22c80,refresh_review,stale_visible,193,1697,down,High: matches both baseline conditions.,The recorded update date may not reflect a rec...


### Weak picks and leakage check

The staleness signal was **OPPOSITE** in this starter snapshot: the 181d+ bucket had a lower observed `down` rate than the 0-30d bucket. Therefore, staleness should not be interpreted as evidence that a page is more likely to be declining.

The visibility signal was **CONFIRMED** directionally: pages with meaningful impressions had higher observed `down` rates than the lowest-impression bucket.

The baseline therefore remains a simple decision-support queue rather than a predictive model. The `stale_visible` rule is useful for identifying pages that combine staleness with meaningful visibility, but the observed data does not establish that staleness itself predicts decline.

A weak pick is a high-ranked row that does not clearly match the intended action or whose observed outcome conflicts with the baseline rationale.

### Leakage check

The baseline score, reason code, action, and ranking use only:
- `days_since_last_update`
- `impressions_90d`

The label-derived fields `trend_direction` and `trend_pct` are not used to calculate the score or ranking. They are used only after ranking for evaluation and review.

No future-window fields or product-decision flags are used by the baseline.

In [21]:
# ML-07 Section 4 — Weak picks + leakage check

print("=== WEAK PICK CHECK ===")

# A weak pick is a high-ranked recommendation whose observed outcome
# does not match the intended decline-oriented review rationale.
weak_picks = top20[
    (top20["reason_code"] == "stale_visible") &
    (top20["trend_direction"] != "down")
].copy()

if len(weak_picks) == 0:
    print("No obvious weak picks found in the top 20.")
else:
    weak_cols = [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_90d",
        "trend_direction",
    ]
    display(weak_picks[weak_cols])

print("\nWeak-pick count:", len(weak_picks))

print("\n=== LEAKAGE CHECK ===")

score_inputs = ["days_since_last_update", "impressions_90d"]

print("Score inputs:", score_inputs)
print("Label fields excluded from score:", ["trend_direction", "trend_pct"])

# Confirm the score is exactly reproducible from the two intended signals.
recomputed_score = (
    (df["days_since_last_update"] >= STALE_DAYS).astype(int)
    + (df["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)
)

score_matches = (df["baseline_score"] == recomputed_score).all()

print("Score uses only the two intended signals:", score_matches)

# Confirm label-derived fields do not appear in the scoring expression.
forbidden_inputs = ["trend_direction", "trend_pct"]

print(
    "No label-derived fields used as score inputs:",
    all(col not in score_inputs for col in forbidden_inputs)
)

# Confirm no future-window fields are used.
future_window_inputs = [
    col for col in score_inputs
    if "last_30" in col or "future" in col or "next" in col
]

print("Future-window inputs found:", future_window_inputs)
print("No future-window inputs used:", len(future_window_inputs) == 0)

print("\n=== SECTION 4 SUMMARY ===")

if len(weak_picks) > 0:
    print(
        "Weak picks are present because the observed outcome does not always "
        "match the refresh-review rationale."
    )
else:
    print("No obvious weak picks were found in the top 20.")

print(
    "The baseline is transparent and uses only pre-decision signal fields; "
    "trend fields are evaluation-only."
)

=== WEAK PICK CHECK ===


,rank,content_id,action,reason_code,days_since_last_update,impressions_90d,trend_direction
23215,12,content_bdbec75c1148,refresh_review,stale_visible,194,1316,stable



Weak-pick count: 1

=== LEAKAGE CHECK ===
Score inputs: ['days_since_last_update', 'impressions_90d']
Label fields excluded from score: ['trend_direction', 'trend_pct']
Score uses only the two intended signals: True
No label-derived fields used as score inputs: True
Future-window inputs found: []
No future-window inputs used: True

=== SECTION 4 SUMMARY ===
Weak picks are present because the observed outcome does not always match the refresh-review rationale.
The baseline is transparent and uses only pre-decision signal fields; trend fields are evaluation-only.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.